# Python-controlled research workflow

This notebook runs Researcher, Fact Checker, and Writer as three explicit Python stages. Every model call uses `gpt-6-luna` with low reasoning effort. Research and fact checking have bounded web-search calls; writing has no tools. The fact checker builds a claim outline with at least three checked sources for each required Xbox topic.

The workflow reads `outcomes.md` beside this notebook and checks the draft requirements before saving. It writes `outputs/citations.json`, `outputs/final_draft.md`, and a compact `outputs/workflow_events.json` containing per-stage usage and estimated cost. Run the cells in order.

## 1. Install dependencies

Run this cell once per fresh kernel.

In [ ]:
%pip install -U openai python-dotenv

## 2. Configure the workflow

Create a `.env` file beside this notebook:

```dotenv
OPENAI_API_KEY=...
RESEARCH_TOPIC=Softmodded Xbox Original Capabilities, video output modes, Region free switching, playing burned discs, auto handling region locked content
```

Keep `outcomes.md` beside the notebook. The workflow reads it at run time and passes its requirements to all three stages. The model is fixed to `gpt-6-luna`. The estimated run budget is $0.10 and the writer targets roughly 900 to 1,300 words. The five required sections are defined in Python below. Update that list if the research topic changes. Cost checks use reported usage after each stage, so the budget is an estimate rather than a prepaid API spending limit.

In [ ]:
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path(".env"))

MODEL = "gpt-6-luna"
TOPIC = os.getenv(
    "RESEARCH_TOPIC",
    "Microsoft Xbox Original Buying Guide - Motherboard revisions, aging issues and what to consider when purchasing",
)
OUTCOMES_PATH = Path("outcomes.md")
if not OUTCOMES_PATH.is_file():
    raise RuntimeError(f"Required acceptance criteria are missing: {OUTCOMES_PATH}")
OUTCOMES_TEXT = OUTCOMES_PATH.read_text(encoding="utf-8").strip()
if not OUTCOMES_TEXT:
    raise RuntimeError(f"Acceptance criteria are empty: {OUTCOMES_PATH}")

OUTPUT_DIR = Path("outputs")
MAX_ESTIMATED_COST_USD = 0.10
MAX_INPUT_CHARS_PER_STAGE = 24_000
MAX_TOTAL_TOKENS = 70_000
MIN_SOURCES_PER_SECTION = 3
SEARCH_LIMITS = {"research": 3, "fact_check": 3, "write": 0}
OUTPUT_LIMITS = {"research": 4_500, "fact_check": 4_500, "write": 6_500}
WORKFLOW_VERSION = "outcomes-v2"
REQUIRED_SECTIONS = {
    "softmod_capabilities": {
        "heading": "Softmod capabilities",
        "research_question": "What capabilities does a softmod actually add, and which depend on a specific tool or setup?",
    },
    "video_output_modes": {
        "heading": "Video output modes",
        "research_question": "Which PAL/NTSC and resolution modes are documented, and what cable, console, and game limits apply?",
    },
    "game_region_switching": {
        "heading": "Game-region switching",
        "research_question": "How are game-region checks handled, and how does changing video standard differ from game region?",
    },
    "burned_discs": {
        "heading": "Playing burned discs",
        "research_question": "What software and optical-drive conditions affect reading burned game discs?",
    },
    "automatic_handling": {
        "heading": "Automatic handling of region-locked content",
        "research_question": "Which behaviors are automatic, which need manual configuration, and how do game and DVD-movie regions differ?",
    },
}

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Missing OPENAI_API_KEY in the environment or .env file.")

print({
    "model": MODEL,
    "topic": TOPIC,
    "workflow_version": WORKFLOW_VERSION,
    "outcomes_path": str(OUTCOMES_PATH),
    "required_sections": list(REQUIRED_SECTIONS),
    "search_limits": SEARCH_LIMITS,
    "estimated_cost_limit_usd": MAX_ESTIMATED_COST_USD,
})

## 3. Define compact stage results

Research returns source-specific evidence for every required topic. The fact checker returns grades, corrected claims, and a sourced outline with at least three directly relevant sources per topic. Python builds numbered Harvard-style reference entries from those records after validating the draft.

In [ ]:
research_reference_schema = {
    "type": "object",
    "properties": {
        "id": {"type": "string"},
        "title": {"type": "string"},
        "url": {"type": "string"},
        "publisher": {"type": "string"},
        "published_date": {"type": "string"},
        "supported_claims": {"type": "array", "items": {"type": "string"}},
        "evidence_summary": {"type": "string"},
        "uncertainty": {"type": "string"},
    },
    "required": ["id", "title", "url", "publisher", "published_date", "supported_claims", "evidence_summary", "uncertainty"],
    "additionalProperties": False,
}

checked_reference_schema = {
    "type": "object",
    "properties": {
        "id": {"type": "string"},
        "title": {"type": "string"},
        "url": {"type": "string"},
        "publisher": {"type": "string"},
        "published_date": {"type": "string"},
        "supported_claims": {"type": "array", "items": {"type": "string"}},
        "evidence_summary": {"type": "string"},
        "grade": {"type": "string", "enum": ["A", "B", "C", "D", "F"]},
        "score": {"type": "integer"},
        "verdict": {"type": "string", "enum": ["verified", "partially_supported", "unverified", "contradicted"]},
        "fact_check_notes": {"type": "string"},
    },
    "required": ["id", "title", "url", "publisher", "published_date", "supported_claims", "evidence_summary", "grade", "score", "verdict", "fact_check_notes"],
    "additionalProperties": False,
}

research_schema = {
    "type": "object",
    "properties": {"references": {"type": "array", "items": research_reference_schema}},
    "required": ["references"],
    "additionalProperties": False,
}

outline_section_schema = {
    "type": "object",
    "properties": {
        "section_key": {"type": "string", "enum": list(REQUIRED_SECTIONS)},
        "key_claim": {"type": "string"},
        "supporting_reference_ids": {"type": "array", "items": {"type": "string"}},
        "caveat": {"type": "string"},
    },
    "required": ["section_key", "key_claim", "supporting_reference_ids", "caveat"],
    "additionalProperties": False,
}

fact_check_schema = {
    "type": "object",
    "properties": {
        "references": {"type": "array", "items": checked_reference_schema},
        "outline": {"type": "array", "items": outline_section_schema},
    },
    "required": ["references", "outline"],
    "additionalProperties": False,
}

## 4. Define one bounded model-call helper

This helper fixes the model and records usage for each stage. Only the research and fact-check stages receive web search. `max_tool_calls` limits built-in tool calls within each response.

In [ ]:
client = OpenAI()
event_log = []
estimated_cost_usd = 0.0

# Current Standard GPT-6 Luna rates in USD per million tokens, plus web search per call.
# Recheck https://developers.openai.com/api/docs/pricing when pricing changes.
TOKEN_RATES = {"input": 0.10, "cached_input": 0.01, "cache_write": 0.125, "output": 0.50}
WEB_SEARCH_RATE = 0.01


def estimate_response_cost(usage, web_search_calls):
    details = usage.input_tokens_details
    cached = (getattr(details, "cached_tokens", 0) or 0) if details else 0
    cache_write = (getattr(details, "cache_write_tokens", 0) or 0) if details else 0
    uncached = max(0, usage.input_tokens - cached - cache_write)
    return (
        uncached * TOKEN_RATES["input"]
        + cached * TOKEN_RATES["cached_input"]
        + cache_write * TOKEN_RATES["cache_write"]
        + usage.output_tokens * TOKEN_RATES["output"]
    ) / 1_000_000 + web_search_calls * WEB_SEARCH_RATE


def run_stage(name, instructions, input_text, *, schema=None):
    global estimated_cost_usd

    if len(input_text) > MAX_INPUT_CHARS_PER_STAGE:
        raise RuntimeError(f"{name} input exceeds {MAX_INPUT_CHARS_PER_STAGE} characters.")

    search_limit = SEARCH_LIMITS[name]
    # Reserve the maximum possible search and output charge before making this call.
    reserved = search_limit * WEB_SEARCH_RATE + OUTPUT_LIMITS[name] * TOKEN_RATES["output"] / 1_000_000
    if estimated_cost_usd + reserved >= MAX_ESTIMATED_COST_USD:
        raise RuntimeError(f"Cost budget would be exceeded before {name}; spent about ${estimated_cost_usd:.4f}.")

    options = {
        "model": MODEL,
        "instructions": instructions,
        "input": input_text,
        "reasoning": {"effort": "low"},
        "max_output_tokens": OUTPUT_LIMITS[name],
        "text": {"verbosity": "medium" if name == "write" else "low"},
        "store": False,
    }
    if schema is not None:
        options["text"]["format"] = {
            "type": "json_schema", "name": name, "strict": True, "schema": schema
        }
    if search_limit:
        options["tools"] = [{"type": "web_search", "search_context_size": "low"}]
        options["tool_choice"] = "required"
        options["max_tool_calls"] = search_limit

    response = client.responses.create(**options)
    web_search_calls = sum(item.type == "web_search_call" for item in response.output)
    if response.usage is None:
        raise RuntimeError(f"{name} returned no usage; cannot enforce the cost check.")

    stage_cost = estimate_response_cost(response.usage, web_search_calls)
    estimated_cost_usd += stage_cost
    event_log.append({
        "stage": name,
        "response_id": response.id,
        "model": MODEL,
        "status": response.status,
        "usage": response.usage.model_dump(mode="json"),
        "web_search_calls": web_search_calls,
        "estimated_cost_usd": round(stage_cost, 6),
    })
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "workflow_events.json").write_text(json.dumps({
        "model": MODEL,
        "workflow_version": WORKFLOW_VERSION,
        "required_sections": list(REQUIRED_SECTIONS),
        "estimated_total_cost_usd": round(estimated_cost_usd, 6),
        "stages": event_log,
    }, indent=2) + "\n", encoding="utf-8")
    print(f"{name}: {response.usage.total_tokens} tokens, {web_search_calls} searches, about ${stage_cost:.4f}")

    if sum(stage["usage"]["total_tokens"] for stage in event_log) > MAX_TOTAL_TOKENS:
        raise RuntimeError(f"Token limit exceeded after {name}; stopping further stages.")
    if estimated_cost_usd > MAX_ESTIMATED_COST_USD:
        raise RuntimeError(f"Cost budget exceeded after {name}: about ${estimated_cost_usd:.4f}.")
    if response.status != "completed" or not response.output_text:
        raise RuntimeError(f"{name} did not complete: {response.status}.")
    return json.loads(response.output_text) if schema is not None else response.output_text

## 5. Run Researcher, Fact Checker, and Writer in Python

The contents of `outcomes.md` and the required topics are passed to research and fact checking. Python requires at least three checked sources per topic before writing. The writer must use the five headings in order, add a Summary, and cite at least three distinct checked sources in every section. Python validates these requirements before writing output files.

In [ ]:
research_input = {
    "topic": TOPIC,
    "required_sections": REQUIRED_SECTIONS,
    "outcomes": OUTCOMES_TEXT,
    "minimum_sources_per_section": MIN_SOURCES_PER_SECTION,
}
research = run_stage(
    "research",
    "Use live web search to find enough independent, materially relevant sources that every required "
    "topic can have at least three distinct directly supporting citations. Prefer primary and authoritative "
    "material. Cover distinctions, limitations, and disagreement. For each source return its exact URL, "
    "publisher, publication date if available, specific supported claims, page-specific evidence summary, "
    "and uncertainty. Do not draft the article. Use IDs R1, R2, and so on. Do not invent source details. "
    "Treat the supplied outcomes as acceptance criteria.",
    json.dumps(research_input, ensure_ascii=False),
    schema=research_schema,
)

if len(research["references"]) < MIN_SOURCES_PER_SECTION:
    raise RuntimeError("Research returned too few sources for the section citation requirement.")

fact_check_input = {
    "research": research,
    "required_sections": REQUIRED_SECTIONS,
    "outcomes": OUTCOMES_TEXT,
    "minimum_sources_per_section": MIN_SOURCES_PER_SECTION,
}
fact_check = run_stage(
    "fact_check",
    "Independently search for or open each source. Check claimed statements, evidence summaries, and circular "
    "sourcing. Keep research IDs and URLs; explain URL corrections in notes. Grade A for primary direct "
    "support, B for authoritative secondary support, C for limitations, D for weak or indirect evidence, "
    "and F for contradicted, inaccessible, or fabricated evidence. Return corrected claims, evidence "
    "summaries, score 0-100, verdict, and concise notes. Return one outline entry per required section. "
    "Each outline entry needs at least three distinct, directly relevant, checked source IDs, a defensible "
    "key claim, and caveat. Use only verified or partially supported A-C sources. If three sources are "
    "unavailable, return the actual smaller list so Python stops before writing. Mark sources you could not "
    "independently check unverified. Apply the supplied outcomes.",
    json.dumps(fact_check_input, ensure_ascii=False),
    schema=fact_check_schema,
)

references = fact_check["references"]
research_ids = {ref["id"] for ref in research["references"]}
checked_ids = {ref["id"] for ref in references}
if checked_ids != research_ids or len(references) != len(research_ids):
    raise RuntimeError("Fact checker omitted or duplicated research IDs.")
if len({ref["url"] for ref in references}) != len(references):
    raise RuntimeError("Fact checker returned duplicate URLs.")
if any(not 0 <= ref["score"] <= 100 for ref in references):
    raise RuntimeError("Fact checker returned a score outside 0-100.")

usable = [
    ref for ref in references
    if ref["grade"] in {"A", "B", "C"}
    and ref["verdict"] in {"verified", "partially_supported"}
]
allowed_ids = {ref["id"] for ref in usable}
if not usable:
    raise RuntimeError("No checked source can support a factual draft.")

outline_by_key = {}
for section in fact_check["outline"]:
    key = section["section_key"]
    if key in outline_by_key:
        raise RuntimeError(f"Fact checker returned duplicate outline section: {key}.")
    outline_by_key[key] = section
missing_keys = set(REQUIRED_SECTIONS) - set(outline_by_key)
extra_keys = set(outline_by_key) - set(REQUIRED_SECTIONS)
if missing_keys or extra_keys:
    raise RuntimeError(
        f"Incomplete outline. Missing: {sorted(missing_keys)}; unexpected: {sorted(extra_keys)}."
    )

outline = []
for key, requirement in REQUIRED_SECTIONS.items():
    section = outline_by_key[key]
    ids = section["supporting_reference_ids"]
    if not section["key_claim"].strip() or len(ids) != len(set(ids)):
        raise RuntimeError(f"Required section {key!r} has no checked claim or has duplicate IDs.")
    if len(ids) < MIN_SOURCES_PER_SECTION or not set(ids) <= allowed_ids:
        raise RuntimeError(
            f"Required section {key!r} needs {MIN_SOURCES_PER_SECTION} distinct checked sources."
        )
    outline.append({**section, "heading": requirement["heading"]})

writer_input = {
    "topic": TOPIC,
    "outline": outline,
    "references": usable,
    "outcomes": OUTCOMES_TEXT,
}
draft_body = run_stage(
    "write",
    "Write a readable article of roughly 900-1,300 words from the checked outline. Use the five supplied "
    "topic headings exactly and in order, then add a Summary heading. Give each topic a distinct scope; "
    "do not repeat the same point in multiple topic sections. Cite at least three different, directly "
    "relevant source IDs from that section's supporting_reference_ids inside each topic section. "
    "Use only single-ID citations such as [R1], never ranges or combined brackets. The Summary should "
    "synthesize the most important points from all five topics without adding new claims, and cite at "
    "least three checked sources. Distinguish facts from caveats. Do not add unsupported claims or "
    "a References section; Python builds numbered Harvard-style references. Apply the supplied outcomes.",
    json.dumps(writer_input, ensure_ascii=False),
).strip()


def validate_draft(draft_text, section_outline, usable_ids):
    """Check structural and source coverage requirements before saving a draft."""
    expected_headings = [section["heading"] for section in section_outline] + ["Summary"]
    actual_headings = re.findall(r"(?m)^## (.+)$", draft_text)
    if actual_headings != expected_headings:
        raise RuntimeError(f"Draft headings do not match required sections: {actual_headings!r}.")
    if re.search(r"(?im)^#+\s*references\b", draft_text):
        raise RuntimeError("Writer added a References section that Python must generate.")

    citation_groups = re.findall(r"\[R\d+[^\]]*\]", draft_text)
    if any(re.fullmatch(r"\[R\d+\]", group) is None for group in citation_groups):
        raise RuntimeError("Citations must use single source IDs such as [R1].")
    cited_ids = set(re.findall(r"\[(R\d+)\]", draft_text))
    if not cited_ids or not cited_ids <= usable_ids:
        raise RuntimeError(f"Draft cites unsupported IDs: {sorted(cited_ids - usable_ids)}")

    sections = {}
    for index, heading in enumerate(expected_headings):
        section_text = draft_text.split(f"## {heading}\n", 1)[1]
        if index + 1 < len(expected_headings):
            section_text = section_text.split(f"## {expected_headings[index + 1]}\n", 1)[0]
        sections[heading] = section_text
        section_ids = set(re.findall(r"\[(R\d+)\]", section_text))
        valid_ids = (
            set(section_outline[index]["supporting_reference_ids"])
            if index < len(section_outline) else usable_ids
        )
        if len(section_ids) < MIN_SOURCES_PER_SECTION or not section_ids <= valid_ids:
            raise RuntimeError(
                f"Draft section {heading!r} needs {MIN_SOURCES_PER_SECTION} distinct relevant citations."
            )
        if not section_text.strip():
            raise RuntimeError(f"Draft section {heading!r} is empty.")

    topic_paragraphs = set()
    for heading in expected_headings[:-1]:
        for paragraph in sections[heading].split("\n\n"):
            normalized = re.sub(r"\s+", " ", paragraph).strip().casefold()
            if len(normalized) < 100:
                continue
            if normalized in topic_paragraphs:
                raise RuntimeError(f"Draft repeats a paragraph in {heading!r}.")
            topic_paragraphs.add(normalized)
    return sections["Summary"].strip()


summary_text = validate_draft(draft_body, outline, allowed_ids)
if len(draft_body.split()) < 700:
    print("Draft is shorter than the requested depth; review before sharing.")

## 6. Save checked citations, the draft, and usage

Python numbers inline citations and formats the reference list with publisher, date, title, URL, and access date. The trace records model, tokens, web calls, and estimated stage cost. It does not contain streaming fragments.

In [ ]:
reference_numbers = {ref["id"]: index for index, ref in enumerate(usable, start=1)}
accessed_date = datetime.now(timezone.utc).date().isoformat()
reference_lines = ["## References"]
for ref in usable:
    year_match = re.search(r"\b(?:19|20)\d{2}\b", ref["published_date"])
    year = year_match.group(0) if year_match else "n.d."
    author = ref["publisher"].strip() or ref["title"].strip()
    reference_lines.append(
        f"- [{reference_numbers[ref['id']]}] {author} ({year}) "
        f"*{ref['title']}*. Available at: {ref['url']} "
        f"(Accessed: {accessed_date})."
    )

numbered_body = re.sub(
    r"\[(R\d+)\]",
    lambda match: f"[{reference_numbers[match.group(1)]}]",
    draft_body,
)
final_draft = numbered_body + "\n\n" + "\n".join(reference_lines) + "\n"
result = {
    "topic": TOPIC,
    "executive_summary": summary_text,
    "final_draft": final_draft,
    "references": references,
    "outline": outline,
    "workflow_version": WORKFLOW_VERSION,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
citations_path = OUTPUT_DIR / "citations.json"
draft_path = OUTPUT_DIR / "final_draft.md"
trace_path = OUTPUT_DIR / "workflow_events.json"

citation_payload = {
    "topic": TOPIC,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model": MODEL,
    "workflow_version": WORKFLOW_VERSION,
    "outcomes": OUTCOMES_TEXT,
    "required_sections": REQUIRED_SECTIONS,
    "outline": outline,
    "references": references,
    "reference_numbers": reference_numbers,
}
citations_path.write_text(
    json.dumps(citation_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
draft_path.write_text(final_draft, encoding="utf-8")
trace_path.write_text(json.dumps({
    "model": MODEL,
    "workflow_version": WORKFLOW_VERSION,
    "estimated_total_cost_usd": round(estimated_cost_usd, 6),
    "stages": event_log,
}, indent=2) + "\n", encoding="utf-8")
print({
    "citations": str(citations_path),
    "draft": str(draft_path),
    "trace": str(trace_path),
    "estimated_cost_usd": round(estimated_cost_usd, 4),
})

## 7. Inspect the result

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result["final_draft"]))
display({"references": len(result["references"]), "estimated_cost_usd": round(estimated_cost_usd, 4)})

## 8. Download local outputs

Run this cell after the workflow completes. It creates a ZIP bundle and displays clickable links for every local artifact.

In [ ]:
from IPython.display import FileLink, display
from zipfile import ZIP_DEFLATED, ZipFile

bundle_path = OUTPUT_DIR / "research_outputs.zip"
download_files = [draft_path, citations_path, trace_path]

with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as bundle:
    for file_path in download_files:
        bundle.write(file_path, arcname=file_path.name)

for file_path in [bundle_path, *download_files]:
    display(FileLink(str(file_path)))